In [1]:
import wandb
api = wandb.Api()
import numpy as np

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
runs = api.runs("ramithx1/Batch-Run-Maximum-Parsimony-A100-META-optimized")

In [4]:
errors = {}

seed = {}

sankoffs = {}
elves = {}

for l in [4,8,16,32,64,128]:
    errors[str(l)] = []
    sankoffs[str(l)] = []
    elves[str(l)] = []
    seed[str(l)] = []
    
    for run in runs:
        if(run.config['n_leaves'] == l and run.config['tLs']==[0, 0.005, 10, 50]):
            
            if("custom" in run.name or 'best cost' not in run.summaryMetrics.keys()):
                continue
                
            if("XAblation" not in run.name):
                continue
            
            # print(run.config['epochs'])
            
            cost = run.summaryMetrics['best cost'] #run.summaryMetrics['traversal cost']
            gt   = run.summaryMetrics['sankoff_cost'] 


            errors[str(l)].append(cost - gt)
            sankoffs[str(l)].append(gt)
            elves[str(l)].append(cost)
            seed[str(l)].append(run.config['seed'])
            

            # print(run.summaryMetrics.keys())
            # print(run.summaryMetrics['sankoff_cost'])
            # print(run.summaryMetrics['traversal cost'])
            # print(run.history)
        


In [5]:
seed

{'4': [51, 50, 49, 48, 47, 46, 45, 44, 43, 42],
 '8': [51, 50, 49, 48, 47, 46, 45, 44, 43, 42],
 '16': [51, 50, 49, 48, 47, 46, 45, 44, 43, 42],
 '32': [51, 50, 49, 48, 47, 46, 45, 44, 43, 42],
 '64': [51, 50, 49, 48, 47, 46, 45, 44, 43, 42],
 '128': [48, 47, 46, 45, 44, 43, 42]}

In [6]:
sankoffs

{'4': [273, 278, 282, 277, 278, 276, 285, 271, 278, 274],
 '8': [652, 656, 656, 650, 648, 646, 668, 645, 662, 648],
 '16': [1404, 1413, 1422, 1398, 1399, 1402, 1421, 1401, 1420, 1396],
 '32': [2913, 2908, 2932, 2907, 2916, 2918, 2916, 2911, 2930, 2903],
 '64': [5927, 5899, 5954, 5948, 5947, 5926, 5929, 5896, 5929, 5938],
 '128': [11966, 11980, 11960, 11968, 11979, 11941, 11963]}

In [7]:
errors

{'4': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 '8': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 '16': [15, 0, 0, 8, 0, 0, 12, 0, 0, 0],
 '32': [57, 69, 73, 37, 45, 55, 52, 105, 58, 74],
 '64': [365, 492, 507, 350, 368, 329, 364, 423, 414, 505],
 '128': [1775, 1772, 1727, 1435, 1538, 1499, 1678]}

In [8]:
elves

{'4': [273, 278, 282, 277, 278, 276, 285, 271, 278, 274],
 '8': [652, 656, 656, 650, 648, 646, 668, 645, 662, 648],
 '16': [1419, 1413, 1422, 1406, 1399, 1402, 1433, 1401, 1420, 1396],
 '32': [2970, 2977, 3005, 2944, 2961, 2973, 2968, 3016, 2988, 2977],
 '64': [6292, 6391, 6461, 6298, 6315, 6255, 6293, 6319, 6343, 6443],
 '128': [13741, 13752, 13687, 13403, 13517, 13440, 13641]}

In [9]:
for key in [4,8,16,32,64,128]:

    err = np.array(errors[str(key)])
    s_gt = np.array(sankoffs[str(key)])
    sol  = np.array(elves[str(key)])
    
    print(key, end = " -> ")
    assert err.shape == sol.shape
    assert s_gt.shape == sol.shape
    print(f" average sankoff = {s_gt.mean()}, avg ans = {sol.mean()}, avg error = {err.mean()}, std = {err.std()}, % error = {err.mean()/s_gt.mean()*100}") 

    

4 ->  average sankoff = 277.2, avg ans = 277.2, avg error = 0.0, std = 0.0, % error = 0.0
8 ->  average sankoff = 653.1, avg ans = 653.1, avg error = 0.0, std = 0.0, % error = 0.0
16 ->  average sankoff = 1407.6, avg ans = 1411.1, avg error = 3.5, std = 5.572252686302013, % error = 0.2486501847115658
32 ->  average sankoff = 2915.4, avg ans = 2977.9, avg error = 62.5, std = 18.06792738528689, % error = 2.1437881594292376
64 ->  average sankoff = 5929.3, avg ans = 6341.0, avg error = 411.7, std = 64.32736587176565, % error = 6.943484053766887
128 ->  average sankoff = 11965.285714285714, avg ans = 13597.285714285714, avg error = 1632.0, std = 129.02048563807887, % error = 13.639457000608903
